
# PharmaLens AI — Target Planning & Regional Target Allocation
### Notebook 20: Smart Annual Target Planning

**Purpose:** Build a data-driven annual target for the next fiscal year and intelligently allocate it across regions.

Supported planning approaches:
- **Top-Down:** company target → regions
- **Bottom-Up:** regional potential → company target
- **Iterative:** top-down constraint + bottom-up opportunity allocation
- **Objective-based:** growth, revenue/value, market-share gain, or market-capture strategy

The notebook produces targets in **Units + Value**, with regional allocation driven by a configurable **Potentiality × Opportunity Score**.

> The notebook is designed to be resilient to different spreadsheet column names. Once the user's actual regional / market-share / RX / molecule files are added, the column mapping cell can be adjusted rather than rewriting the model.



## 1. Required / Recommended Data

### Minimum
1. Historical company/brand sales by region:
   - Region
   - Year / Date
   - Units
   - Value

### Strongly recommended
2. **Market Share by region** — enables share-gap and market-capture logic.
3. **RX by region** — enables demand/prescription potential.
4. **Molecule performance** — enables molecule-level opportunity and portfolio mix.

### Very useful additions
5. Region population / HCP count / hospital count.
6. Distribution / numeric distribution / outlet coverage.
7. Current-year YTD sales or latest forecast.
8. Historical regional targets vs actual achievement.
9. Price / ASP history by region or product.
10. Strategic priority or management override by region.

The notebook can run without all optional datasets. Missing inputs are explicitly flagged and their weights are redistributed.


In [ ]:

# ============================================================
# 2. Configuration
# ============================================================
from pathlib import Path
import pandas as pd
import numpy as np

PROJECT_ROOT = Path("..")
DATA_DIR = PROJECT_ROOT / "data"

# Put your files here when available.
FILES = {
    "sales": DATA_DIR / "regional_sales.xlsx",
    "market_share": DATA_DIR / "market_share_region.xlsx",
    "rx": DATA_DIR / "rx_per_region.xlsx",
    "molecules": DATA_DIR / "molecules.xlsx",
}

# Planning year
TARGET_YEAR = 2027
BASE_YEAR = 2026

# Target objective:
# "growth" | "value" | "market_share" | "market_capture" | "custom"
OBJECTIVE = "growth"

# Planning method:
# "top_down" | "bottom_up" | "iterative"
METHOD = "iterative"

# Company-level target assumptions
BASE_VALUE = None       # if None -> calculated from latest sales year
BASE_UNITS = None       # if None -> calculated from latest sales year

TARGET_GROWTH = 0.10    # e.g. 10% growth
TARGET_VALUE = None     # optional explicit company value target
TARGET_UNITS = None     # optional explicit company unit target

# Market-capture strategy
PROJECTED_MARKET_GROWTH = 0.08
CURRENT_MARKET_SHARE = None
DESIRED_MARKET_SHARE = None

# ASP assumption if next-year price is not supplied
DEFAULT_PRICE_GROWTH = 0.03

# Potentiality / opportunity weights
WEIGHTS = {
    "market_potential": 0.25,
    "growth": 0.20,
    "share_gap": 0.15,
    "rx_potential": 0.15,
    "molecule_opportunity": 0.10,
    "execution_strength": 0.10,
    "strategic_priority": 0.05,
}

# Safety / realism controls
MAX_REGIONAL_GROWTH = 0.35
MIN_REGIONAL_GROWTH = -0.10
DAMPING_POWER = 0.70

# Optional management overrides by region.
# Example: {"Central": 1.10, "Western": 0.95}
REGION_OVERRIDE = {}

print("Configuration loaded.")


In [ ]:

# ============================================================
# 3. Flexible Excel Loader + Column Detection
# ============================================================
import re

ALIASES = {
    "region": ["region", "regions", "sales region", "area", "territory", "zone"],
    "year": ["year", "fiscal year", "fy", "date", "period"],
    "units": ["units", "unit", "volume", "sales units", "quantity", "qty"],
    "value": ["value", "sales value", "sales", "revenue", "net sales", "amount"],
    "market_share": ["market share", "share", "ms", "market_share"],
    "rx": ["rx", "prescriptions", "prescription", "trx", "nrx"],
    "molecule": ["molecule", "molecule name", "ingredient", "active ingredient"],
    "brand": ["brand", "brand name", "product", "product name"],
    "growth": ["growth", "growth rate", "yoy growth", "yoy"],
    "population": ["population", "pop"],
    "hcp_count": ["hcp count", "hcps", "hcp", "doctors", "physicians"],
    "priority": ["priority", "strategic priority", "priority score"],
    "achievement": ["achievement", "target achievement", "attainment"],
    "price": ["price", "asp", "average price", "avg price"],
}

def norm(x):
    return re.sub(r"[^a-z0-9]+", "_", str(x).strip().lower()).strip("_")

def detect_columns(df):
    normalized = {norm(c): c for c in df.columns}
    mapping = {}
    for key, aliases in ALIASES.items():
        candidates = [norm(a) for a in aliases]
        for c_norm, original in normalized.items():
            if c_norm in candidates:
                mapping[key] = original
                break
    return mapping

def load_any_excel(path):
    if path is None or not Path(path).exists():
        return None
    xls = pd.ExcelFile(path)
    sheets = {}
    for sheet in xls.sheet_names:
        df = pd.read_excel(path, sheet_name=sheet)
        if not df.empty:
            sheets[sheet] = df
    return sheets

loaded = {}
for name, path in FILES.items():
    loaded[name] = load_any_excel(path)

for name, sheets in loaded.items():
    if sheets:
        print(f"{name}: {list(sheets.keys())}")
    else:
        print(f"{name}: not supplied yet")


In [ ]:

# ============================================================
# 4. Select the Most Relevant Sheet Automatically
# ============================================================
def select_sheet(sheets, required_keys=("region",)):
    if not sheets:
        return None, None
    best = None
    best_score = -1
    for sheet_name, df in sheets.items():
        mapping = detect_columns(df)
        score = sum(k in mapping for k in required_keys)
        if score > best_score:
            best_score = score
            best = (sheet_name, df, mapping)
    return best if best else (None, None, {})

sales_sheet, sales_raw, sales_map = select_sheet(
    loaded["sales"], required_keys=("region", "units", "value")
)

market_sheet, market_raw, market_map = select_sheet(
    loaded["market_share"], required_keys=("region", "market_share")
)

rx_sheet, rx_raw, rx_map = select_sheet(
    loaded["rx"], required_keys=("region", "rx")
)

mol_sheet, mol_raw, mol_map = select_sheet(
    loaded["molecules"], required_keys=("region", "molecule")
)

print("Detected mappings:")
print("Sales:", sales_map)
print("Market Share:", market_map)
print("RX:", rx_map)
print("Molecules:", mol_map)



## 5. Planning Logic

The core model creates a **Regional Potentiality × Opportunity Score**.

### Regional Opportunity Score

Each region is scored on normalized components:

- **Market Potential:** size of current market / addressable market
- **Growth:** historical or forecast growth
- **Share Gap:** room to gain share versus the strategic share ambition
- **RX Potential:** prescription demand signal
- **Molecule Opportunity:** portfolio/molecule growth and attractiveness
- **Execution Strength:** historical target achievement / sales momentum
- **Strategic Priority:** optional management input

The final score is a weighted composite.

`Opportunity Score = Σ(weight_i × normalized factor_i)`

The target allocation then applies a **damping function** so that the highest-potential region does not automatically receive an unrealistic amount of the target.

This is intentionally more sophisticated than simply allocating target according to last year's sales share.


In [ ]:

# ============================================================
# 6. Standardize Sales Data
# ============================================================
def clean_sales(df, mapping):
    if df is None:
        return None

    out = df.copy()
    rename = {}
    for standard, original in mapping.items():
        rename[original] = standard
    out = out.rename(columns=rename)

    if "region" not in out.columns:
        raise ValueError("Sales file must contain a Region column.")

    for c in ["units", "value"]:
        if c in out.columns:
            out[c] = pd.to_numeric(out[c], errors="coerce")

    if "year" in out.columns:
        if pd.api.types.is_datetime64_any_dtype(out["year"]):
            out["year"] = out["year"].dt.year
        else:
            out["year"] = pd.to_numeric(
                out["year"].astype(str).str.extract(r"(20\d{2})")[0],
                errors="coerce"
            )

    return out

sales = clean_sales(sales_raw, sales_map)

if sales is not None:
    if "year" in sales.columns:
        sales = sales[sales["year"].notna()].copy()
    print(sales.head())
else:
    print("Sales data not supplied. Add the file and rerun this section.")


In [ ]:

# ============================================================
# 7. Build Regional Historical Base
# ============================================================
def aggregate_latest_sales(sales):
    if sales is None:
        return None

    if "year" in sales.columns:
        latest_year = int(sales["year"].max())
        base = sales[sales["year"] == latest_year].copy()
    else:
        latest_year = None
        base = sales.copy()

    agg = base.groupby("region", dropna=False).agg(
        base_units=("units", "sum") if "units" in base else ("region", "size"),
        base_value=("value", "sum") if "value" in base else ("region", "size"),
    ).reset_index()

    agg["asp"] = np.where(
        agg["base_units"] > 0,
        agg["base_value"] / agg["base_units"],
        np.nan
    )
    return agg, latest_year

if sales is not None:
    regional_base, detected_base_year = aggregate_latest_sales(sales)
    print("Detected base year:", detected_base_year)
    display(regional_base.sort_values("base_value", ascending=False))
else:
    regional_base = None


In [ ]:

# ============================================================
# 8. Historical Growth by Region
# ============================================================
def calculate_growth(sales):
    if sales is None or "year" not in sales.columns:
        return pd.DataFrame(columns=["region", "growth"])

    yearly = sales.groupby(["region", "year"], dropna=False).agg(
        value=("value", "sum"),
        units=("units", "sum")
    ).reset_index().sort_values(["region", "year"])

    yearly["value_growth"] = yearly.groupby("region")["value"].pct_change()
    yearly["unit_growth"] = yearly.groupby("region")["units"].pct_change()

    growth = yearly.groupby("region").tail(1)[
        ["region", "value_growth", "unit_growth"]
    ].copy()

    growth["growth"] = growth["value_growth"].fillna(growth["unit_growth"])
    return growth

growth_df = calculate_growth(sales)
display(growth_df)


In [ ]:

# ============================================================
# 9. Add Market Share, RX and Molecule Signals
# ============================================================
def aggregate_market_share(df, mapping):
    if df is None or "region" not in mapping or "market_share" not in mapping:
        return pd.DataFrame(columns=["region", "market_share"])

    x = df.rename(columns={mapping["region"]: "region",
                           mapping["market_share"]: "market_share"}).copy()
    x["market_share"] = pd.to_numeric(x["market_share"], errors="coerce")

    # If multiple rows exist, use weighted/mean signal depending on data shape.
    return x.groupby("region", dropna=False)["market_share"].mean().reset_index()

def aggregate_rx(df, mapping):
    if df is None or "region" not in mapping or "rx" not in mapping:
        return pd.DataFrame(columns=["region", "rx"])

    x = df.rename(columns={mapping["region"]: "region",
                           mapping["rx"]: "rx"}).copy()
    x["rx"] = pd.to_numeric(x["rx"], errors="coerce")
    return x.groupby("region", dropna=False)["rx"].sum().reset_index()

def aggregate_molecule_opportunity(df, mapping):
    if df is None or "region" not in mapping:
        return pd.DataFrame(columns=["region", "molecule_opportunity"])

    x = df.rename(columns={mapping["region"]: "region"}).copy()

    if "growth" in mapping:
        x = x.rename(columns={mapping["growth"]: "growth"})
        x["growth"] = pd.to_numeric(x["growth"], errors="coerce")
        score = x.groupby("region")["growth"].mean().reset_index()
        score = score.rename(columns={"growth": "molecule_opportunity"})
    elif "value" in mapping:
        x = x.rename(columns={mapping["value"]: "value"})
        x["value"] = pd.to_numeric(x["value"], errors="coerce")
        score = x.groupby("region")["value"].sum().reset_index()
        score = score.rename(columns={"value": "molecule_opportunity"})
    else:
        score = x.groupby("region").size().reset_index(name="molecule_opportunity")

    return score

ms_df = aggregate_market_share(market_raw, market_map)
rx_df = aggregate_rx(rx_raw, rx_map)
mol_df = aggregate_molecule_opportunity(mol_raw, mol_map)

print("Market share rows:", len(ms_df))
print("RX rows:", len(rx_df))
print("Molecule opportunity rows:", len(mol_df))


In [ ]:

# ============================================================
# 10. Build Regional Potentiality / Opportunity Model
# ============================================================
def minmax(s):
    s = pd.to_numeric(s, errors="coerce")
    if s.notna().sum() == 0:
        return pd.Series(0.5, index=s.index)
    lo, hi = s.min(), s.max()
    if pd.isna(lo) or pd.isna(hi) or hi == lo:
        return pd.Series(0.5, index=s.index)
    return (s - lo) / (hi - lo)

def build_opportunity_table():
    if regional_base is None:
        raise ValueError("Regional sales data is required to calculate regional targets.")

    df = regional_base.copy()

    for extra in [growth_df, ms_df, rx_df, mol_df]:
        if not extra.empty:
            df = df.merge(extra, on="region", how="left")

    # Market potential proxy = base value.
    df["market_potential_raw"] = df["base_value"]

    # Growth signal
    df["growth_raw"] = df.get("growth", pd.Series(np.nan, index=df.index))

    # Share gap:
    if "market_share" in df.columns and DESIRED_MARKET_SHARE is not None:
        df["share_gap_raw"] = DESIRED_MARKET_SHARE - df["market_share"]
    else:
        # If no explicit ambition, regions with lower share get more whitespace.
        df["share_gap_raw"] = 1 - df.get("market_share", pd.Series(np.nan, index=df.index))

    # RX potential proxy
    df["rx_potential_raw"] = df.get("rx", pd.Series(np.nan, index=df.index))

    # Molecule signal
    df["molecule_opportunity_raw"] = df.get(
        "molecule_opportunity", pd.Series(np.nan, index=df.index)
    )

    # Execution strength: growth is the fallback if achievement isn't available.
    if "achievement" in df.columns:
        df["execution_strength_raw"] = df["achievement"]
    else:
        df["execution_strength_raw"] = df["growth_raw"]

    # Strategic priority defaults to neutral.
    df["strategic_priority_raw"] = 1.0

    for region, multiplier in REGION_OVERRIDE.items():
        df.loc[df["region"].astype(str).eq(str(region)), "strategic_priority_raw"] *= multiplier

    components = {
        "market_potential": "market_potential_raw",
        "growth": "growth_raw",
        "share_gap": "share_gap_raw",
        "rx_potential": "rx_potential_raw",
        "molecule_opportunity": "molecule_opportunity_raw",
        "execution_strength": "execution_strength_raw",
        "strategic_priority": "strategic_priority_raw",
    }

    available_weight = 0
    normalized = {}
    for key, col in components.items():
        if col in df.columns and df[col].notna().sum() >= 2:
            normalized[key] = minmax(df[col]).fillna(0.5)
            available_weight += WEIGHTS[key]
        else:
            normalized[key] = pd.Series(0.5, index=df.index)

    # Redistribute unavailable weights across available factors.
    effective_weights = {
        k: (WEIGHTS[k] / available_weight if available_weight > 0 and components[k] in df.columns
            and df[components[k]].notna().sum() >= 2 else 0)
        for k in components
    }

    df["opportunity_score"] = 0
    for key in components:
        df[key + "_score"] = normalized[key]
        df["opportunity_score"] += effective_weights[key] * normalized[key]

    # Damping prevents a highly ranked region from absorbing a disproportionate target.
    df["allocation_factor"] = np.power(
        np.clip(df["opportunity_score"], 0.05, 1.0),
        DAMPING_POWER
    )

    return df, effective_weights

opportunity, effective_weights = build_opportunity_table()

print("Effective weights:", effective_weights)
display(
    opportunity[
        ["region", "base_value", "base_units", "opportunity_score", "allocation_factor"]
    ].sort_values("opportunity_score", ascending=False)
)



## 11. Company Target Engine

The company target can come from several strategic routes:

### A. Growth target
`Target Value = Base Value × (1 + Growth %)`

### B. Market-share capture
1. Forecast total market.
2. Apply desired company market share.
3. Calculate required company value.

### C. Explicit management target
Use `TARGET_VALUE` and/or `TARGET_UNITS`.

### D. Bottom-up
Estimate each region independently and sum the regional targets.

### E. Iterative
Use a top-down company target as the hard constraint, then allocate it according to regional opportunity. This is the recommended default when management already has a corporate ambition but wants a smarter regional split.


In [ ]:

# ============================================================
# 12. Company Target Engine
# ============================================================
def company_base_metrics():
    if regional_base is None:
        raise ValueError("Regional base is unavailable.")
    return regional_base["base_value"].sum(), regional_base["base_units"].sum()

def determine_company_target():
    base_value, base_units = company_base_metrics()

    target_value = TARGET_VALUE
    target_units = TARGET_UNITS

    if OBJECTIVE == "market_capture" and DESIRED_MARKET_SHARE is not None:
        if CURRENT_MARKET_SHARE is None:
            raise ValueError("CURRENT_MARKET_SHARE is required for market_capture objective.")

        projected_market_value = base_value * (
            (1 + PROJECTED_MARKET_GROWTH) / max(CURRENT_MARKET_SHARE, 1e-9)
        )
        target_value = projected_market_value * DESIRED_MARKET_SHARE

    elif target_value is None:
        target_value = base_value * (1 + TARGET_GROWTH)

    if target_units is None:
        implied_asp = base_value / base_units if base_units > 0 else np.nan
        target_units = target_value / (
            implied_asp * (1 + DEFAULT_PRICE_GROWTH)
        ) if implied_asp > 0 else base_units * (1 + TARGET_GROWTH)

    return {
        "base_value": base_value,
        "base_units": base_units,
        "target_value": target_value,
        "target_units": target_units,
        "value_growth": target_value / base_value - 1 if base_value else np.nan,
        "unit_growth": target_units / base_units - 1 if base_units else np.nan,
    }

company_target = determine_company_target()
company_target


In [ ]:

# ============================================================
# 13. Bottom-Up Regional Potential Target
# ============================================================
def bottom_up_target(df):
    x = df.copy()

    # Base regional growth expectation combines historical growth with the global target.
    hist_growth = x["growth_raw"].fillna(TARGET_GROWTH).clip(
        MIN_REGIONAL_GROWTH, MAX_REGIONAL_GROWTH
    )

    # Opportunity premium: high-potential regions receive more upside.
    opp_centered = x["opportunity_score"] - x["opportunity_score"].mean()
    regional_growth = hist_growth + 0.20 * opp_centered
    regional_growth = regional_growth.clip(MIN_REGIONAL_GROWTH, MAX_REGIONAL_GROWTH)

    x["bottom_up_value_target"] = x["base_value"] * (1 + regional_growth)

    # Price assumption -> unit target
    x["projected_asp"] = x["asp"].fillna(
        x["base_value"].sum() / max(x["base_units"].sum(), 1)
    ) * (1 + DEFAULT_PRICE_GROWTH)

    x["bottom_up_units_target"] = (
        x["bottom_up_value_target"] / x["projected_asp"]
    )

    return x

bottom_up = bottom_up_target(opportunity)

display(
    bottom_up[
        ["region", "base_value", "growth_raw", "opportunity_score",
         "bottom_up_value_target", "bottom_up_units_target"]
    ].sort_values("bottom_up_value_target", ascending=False)
)


In [ ]:

# ============================================================
# 14. Smart Allocation — Top-Down / Iterative
# ============================================================
def constrained_allocate(base, weights, total_target, min_growth=MIN_REGIONAL_GROWTH,
                          max_growth=MAX_REGIONAL_GROWTH):
    base = np.asarray(base, dtype=float)
    weights = np.asarray(weights, dtype=float)

    # Initial opportunity allocation
    weights = np.clip(weights, 1e-9, None)
    weights = weights / weights.sum()
    raw = total_target * weights

    # Iterative clipping around realistic growth limits.
    lower = base * (1 + min_growth)
    upper = base * (1 + max_growth)
    target = raw.copy()

    for _ in range(100):
        low_violation = target < lower
        high_violation = target > upper

        target[low_violation] = lower[low_violation]
        target[high_violation] = upper[high_violation]

        fixed = low_violation | high_violation
        remainder = total_target - target[fixed].sum()

        if remainder <= 0:
            break

        free = ~fixed
        if not free.any():
            break

        free_weights = weights[free]
        free_weights = free_weights / free_weights.sum()
        target[free] = remainder * free_weights

    # Final proportional reconciliation.
    if target.sum() > 0:
        target *= total_target / target.sum()

    return target

def allocate_targets(method=METHOD):
    x = opportunity.copy()

    if method == "bottom_up":
        out = bottom_up.copy()
        # Reconcile bottom-up value to company objective.
        value_scale = company_target["target_value"] / out["bottom_up_value_target"].sum()
        out["target_value"] = out["bottom_up_value_target"] * value_scale
        out["target_units"] = out["target_value"] / out["projected_asp"]
        return out

    # Top-down and iterative both start from company target.
    weights = x["allocation_factor"].values

    if method == "top_down":
        out = x.copy()
        out["target_value"] = constrained_allocate(
            out["base_value"].values, weights, company_target["target_value"]
        )
    else:
        # Iterative: blend opportunity allocation with bottom-up recommendation.
        bu = bottom_up.copy()
        bu_weight = bu["bottom_up_value_target"].values
        bu_weight = np.clip(bu_weight, 1e-9, None)
        bu_weight = bu_weight / bu_weight.sum()

        opp_weight = weights / np.sum(weights)

        blended_weight = 0.55 * opp_weight + 0.45 * bu_weight

        out = x.copy()
        out["target_value"] = constrained_allocate(
            out["base_value"].values,
            blended_weight,
            company_target["target_value"]
        )

    # Projected ASP and units.
    out["projected_asp"] = out["asp"].fillna(
        company_target["base_value"] / max(company_target["base_units"], 1)
    ) * (1 + DEFAULT_PRICE_GROWTH)

    out["target_units"] = out["target_value"] / out["projected_asp"]

    # Actual implied regional growth.
    out["value_growth_target"] = out["target_value"] / out["base_value"] - 1
    out["unit_growth_target"] = out["target_units"] / out["base_units"] - 1

    return out

targets = allocate_targets(METHOD)

display(
    targets[
        ["region", "base_value", "base_units", "opportunity_score",
         "target_value", "target_units", "value_growth_target",
         "unit_growth_target"]
    ].sort_values("target_value", ascending=False)
)


In [ ]:

# ============================================================
# 15. Target Quality / Sanity Checks
# ============================================================
def target_quality_report(df):
    checks = {
        "Company value target": df["target_value"].sum(),
        "Required company value target": company_target["target_value"],
        "Value reconciliation error": df["target_value"].sum() - company_target["target_value"],
        "Company units target": df["target_units"].sum(),
        "Required company units target": company_target["target_units"],
        "Regions": df["region"].nunique(),
        "Highest regional value growth": df["value_growth_target"].max(),
        "Lowest regional value growth": df["value_growth_target"].min(),
        "Highest opportunity score": df["opportunity_score"].max(),
        "Lowest opportunity score": df["opportunity_score"].min(),
    }
    return pd.Series(checks)

quality = target_quality_report(targets)
display(quality)

assert abs(quality["Value reconciliation error"]) < max(
    1.0, company_target["target_value"] * 0.001
), "Target value did not reconcile to the company target."


In [ ]:

# ============================================================
# 16. Management-Friendly Output
# ============================================================
final_columns = [
    "region",
    "base_value",
    "base_units",
    "growth_raw",
    "market_share" if "market_share" in targets.columns else "region",
    "rx" if "rx" in targets.columns else "region",
    "opportunity_score",
    "target_value",
    "target_units",
    "value_growth_target",
    "unit_growth_target",
]

final_columns = list(dict.fromkeys([c for c in final_columns if c in targets.columns]))
regional_target_output = targets[final_columns].copy()

regional_target_output = regional_target_output.sort_values(
    "target_value", ascending=False
)

# Company summary
company_summary = pd.DataFrame([{
    "base_year": BASE_YEAR,
    "target_year": TARGET_YEAR,
    "method": METHOD,
    "objective": OBJECTIVE,
    "base_value": company_target["base_value"],
    "target_value": company_target["target_value"],
    "value_growth": company_target["value_growth"],
    "base_units": company_target["base_units"],
    "target_units": company_target["target_units"],
    "unit_growth": company_target["unit_growth"],
}])

display(company_summary)
display(regional_target_output)


In [ ]:

# ============================================================
# 17. Export Results
# ============================================================
OUTPUT_DIR = PROJECT_ROOT / "outputs"
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

output_file = OUTPUT_DIR / f"PharmaLens_Target_Plan_{TARGET_YEAR}.xlsx"

with pd.ExcelWriter(output_file, engine="openpyxl") as writer:
    company_summary.to_excel(writer, sheet_name="Company_Target", index=False)
    regional_target_output.to_excel(writer, sheet_name="Regional_Targets", index=False)
    opportunity.to_excel(writer, sheet_name="Opportunity_Model", index=False)
    quality.to_frame("Value").to_excel(writer, sheet_name="Quality_Checks")

print(f"Saved: {output_file}")



## 18. Recommended Next Upgrade

When the real files are added, this notebook should be extended with:

1. **Region × Molecule target allocation** — not just Region.
2. **Region × Brand target allocation** — for field-force execution.
3. **Market forecast by region** — instead of one national market-growth assumption.
4. **Market-share bridge** — current share → target share → required units/value.
5. **RX-to-sales conversion model** — estimate expected sales from prescription potential.
6. **Capacity constraint** — reps, HCP coverage, distributor capacity and inventory.
7. **Scenario engine** — Conservative / Base / Stretch.
8. **Target feasibility score** — probability of achieving each regional target.
9. **Target vs Budget linkage** — connect commercial target to the existing PharmaLens Budget module.
10. **Quarterly phasing** — Q1/Q2/Q3/Q4 targets using seasonality.
11. **Management override loop** — allow leadership to adjust selected regions while automatically rebalancing the remaining regions.
12. **Explainable target rationale** — every region receives a human-readable explanation of why its target is high/low.

The intended production version should therefore become a **Target Planning Engine**, not merely a target calculator.
